# GraphFleet Advanced Querying Guide

This notebook demonstrates advanced querying features in GraphFleet, following GraphRAG's querying approach. We'll cover:

1. Query types and strategies
2. Context window management
3. Query parameters tuning
4. Advanced retrieval techniques

For more details, see [GraphRAG Query Documentation](https://microsoft.github.io/graphrag/query/overview/)

In [ ]:
import os
from pathlib import Path
from graphfleet.core import GraphFleet
from graphfleet.querying import QueryOptimizer

# Initialize project
project_dir = Path("./data")
graph_fleet = GraphFleet(project_dir)

## 1. Query Types

GraphFleet supports various query types from GraphRAG:

In [ ]:
# Standard query - balances local and global context
standard_result = await graph_fleet.query(
    "What are the main applications of machine learning?",
    query_type="standard"
)

# Local query - focuses on local document context
local_result = await graph_fleet.query(
    "What is supervised learning?",
    query_type="local"
)

# Global query - considers broader knowledge graph
global_result = await graph_fleet.query(
    "How do different AI technologies relate to each other?",
    query_type="global"
)

# Print results
print("Standard Query Result:")
print(standard_result)
print("\nLocal Query Result:")
print(local_result)
print("\nGlobal Query Result:")
print(global_result)

## 2. Context Window Management

Control how much context is used for queries:

In [ ]:
# Query with different context windows
narrow_context = await graph_fleet.query(
    "What is deep learning?",
    context_window=2,     # Use 2 chunks of context
    max_tokens=1000       # Limit response tokens
)

wide_context = await graph_fleet.query(
    "What is deep learning?",
    context_window=5,     # Use 5 chunks of context
    max_tokens=1000       # Limit response tokens
)

print("Narrow Context Result:")
print(narrow_context)
print("\nWide Context Result:")
print(wide_context)

## 3. Query Parameters Tuning

Optimize query parameters for better results:

In [ ]:
# Initialize query optimizer
optimizer = QueryOptimizer(
    evaluation_model="gpt-4",
    num_trials=5
)

# Define parameter space
param_space = {
    "temperature": (0.0, 1.0),
    "top_k": (3, 10),
    "similarity_threshold": (0.5, 0.9)
}

# Optimize parameters
best_params = await optimizer.optimize(
    query="What are the differences between AI and ML?",
    param_space=param_space
)

print("Best Parameters:")
print(best_params)

# Query with optimized parameters
optimized_result = await graph_fleet.query(
    "What are the differences between AI and ML?",
    **best_params
)

print("\nOptimized Query Result:")
print(optimized_result)

## 4. Advanced Retrieval Techniques

Explore different retrieval strategies:

In [ ]:
# Hybrid retrieval
hybrid_result = await graph_fleet.query(
    "How does neural network training work?",
    retrieval_type="hybrid",
    vector_weight=0.7,    # Weight for vector similarity
    keyword_weight=0.3    # Weight for keyword matching
)

# Multi-hop retrieval
multihop_result = await graph_fleet.query(
    "How do CNNs relate to computer vision applications?",
    retrieval_type="multihop",
    max_hops=2,          # Maximum number of hops
    min_relevance=0.6    # Minimum relevance score
)

print("Hybrid Retrieval Result:")
print(hybrid_result)
print("\nMulti-hop Retrieval Result:")
print(multihop_result)

## 5. Query Analysis and Debugging

Analyze and debug query results:

In [ ]:
# Query with analysis
result = await graph_fleet.query(
    "What are the applications of reinforcement learning?",
    return_analysis=True  # Return detailed analysis
)

print("Query Analysis:")
print("\nRetrieved Chunks:")
for chunk in result["analysis"]["chunks"]:
    print(f"- Score: {chunk['score']:.2f}")
    print(f"- Content: {chunk['content'][:100]}...")

print("\nReasoning Steps:")
for step in result["analysis"]["reasoning"]:
    print(f"- {step}")

print("\nConfidence Scores:")
print(result["analysis"]["confidence"])

## 6. Custom Query Pipelines

Create custom query pipelines for specific use cases:

In [ ]:
async def technical_query_pipeline(query: str):
    """Custom pipeline for technical documentation queries."""
    # Step 1: Get broad context
    global_context = await graph_fleet.query(
        query,
        query_type="global",
        max_results=5
    )
    
    # Step 2: Get specific details
    local_context = await graph_fleet.query(
        query,
        query_type="local",
        context_window=2
    )
    
    # Step 3: Combine and synthesize
    combined_query = f"""
    Based on:
    1. Broad context: {global_context}
    2. Specific details: {local_context}
    
    Provide a comprehensive answer to: {query}
    """
    
    final_result = await graph_fleet.query(
        combined_query,
        temperature=0.3  # Lower temperature for more focused response
    )
    
    return final_result

# Try the custom pipeline
result = await technical_query_pipeline(
    "How do transformers work in deep learning?"
)

print("Custom Pipeline Result:")
print(result)